# Qwen3Loop-0.6B: Recursive Intermediate-Calibrated Tool GRPO (RIC-GRPO)
### Treinamento Distribuído em Dual GPU (2x Tesla T4 16GB)

> **Status:** Configurado com DDP, precisão float16 otimizada, mmap bfloat16 instantâneo, compatibilidade universal de transformers.

In [ ]:
# 1. Verificar GPUs Tesla T4 ativas
!nvidia-smi

In [ ]:
# 2. Instalar dependências e remover torchao obsoleto da imagem base
!pip uninstall -y torchao
!pip install -q transformers peft datasets accelerate

In [ ]:
# 3. Preparar e copiar diretório de trabalho
import os, sys, shutil

print('Escaneando /kaggle/input recursivamente...')
INPUT_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'rewards_tool_calling.py' in files and 'config.json' in files:
        INPUT_DIR = root
        break

if INPUT_DIR is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'model.safetensors' in files:
            INPUT_DIR = os.path.dirname(root)
            break

print('Diretório de entrada detectado com sucesso:', INPUT_DIR)
assert INPUT_DIR is not None, f'Falha ao localizar os arquivos em /kaggle/input'

# Copiar engine para o sys.path do Kaggle
os.makedirs('/kaggle/working/engine/qwen3loop_python', exist_ok=True)
shutil.copytree(os.path.join(INPUT_DIR, 'engine', 'qwen3loop_python'), '/kaggle/working/engine/qwen3loop_python', dirs_exist_ok=True)
shutil.copy2(os.path.join(INPUT_DIR, 'rewards_tool_calling.py'), '/kaggle/working/rewards_tool_calling.py')

os.environ['MODEL_DIR'] = os.path.join(INPUT_DIR, 'model')
os.environ['CURRICULUM_DIR'] = os.path.join(INPUT_DIR, 'data')
os.environ['OUTPUT_DIR'] = '/kaggle/working/qwen3loop_tool_grpo_final'

print('Ambiente e dependências preparados com sucesso!')

In [ ]:
%%writefile /kaggle/working/engine/qwen3loop_python/modeling_qwen3loop.py
# coding=utf-8
# Copyright 2026 Qwen3Loop team and HuggingFace Inc. team.
# Licensed under the Apache License, Version 2.0 (the "License");

""" Qwen3Loop Model Architecture - Looped Transformer for Qwen3 via Direct HF Inheritance

Paridade de arquitetura com o Nanbeige4.2-3B (modeling_nanbeige.py). O laço de
loop, a indexação virtual do KV cache e a norm entre passagens seguem o mesmo
esquema; os defaults reproduzem os pesos publicados do Nanbeige4.2-3B, que NÃO
ativam LoopSplit, mHC, depth attention nem n-gram embeddings.
"""

from typing import Dict, List, Optional, Tuple, Union
import torch
from torch import nn
from transformers import AutoConfig, AutoModelForCausalLM
from transformers.models.qwen3.modeling_qwen3 import (
    Qwen3Model,
    Qwen3ForCausalLM,
    create_causal_mask,
    create_sliding_window_causal_mask,
)
from transformers.cache_utils import Cache, DynamicCache
from transformers.modeling_outputs import BaseModelOutputWithPast, CausalLMOutputWithPast

from .configuration_qwen3loop import Qwen3LoopConfig


def get_loop_cache_layer_idx(layer_idx: int, loop_idx: int, num_hidden_layers: int,
                             cache_layer_idx: Optional[int] = None) -> int:
    """Índice virtual de camada no KV cache. Igual a _get_loop_cache_layer_idx do Nanbeige."""
    if cache_layer_idx is not None:
        return cache_layer_idx
    return layer_idx + loop_idx * num_hidden_layers


def get_double_loop_split_layer_order(num_hidden_layers: int,
                                      loop_middle_layers: Optional[int] = None
                                      ) -> List[Tuple[int, Optional[int]]]:
    """
    LoopSplit. Porte fiel de _get_double_loop_split_layer_order_with_mhc_loop_indices.

    Mantém um prefixo e um sufixo executados uma única vez e repete só o miolo.
    Para N=22, M=11: [0..4] uma vez, [5..15] tres vezes, [16..21] uma vez = 44
    execuções — mesmo custo de 2 loops completos, distribuído de outra forma.

    Devolve pares (layer_idx, repeat_idx), com repeat_idx=None nas camadas não iteradas.
    """
    if num_hidden_layers <= 0:
        raise ValueError("enable_double_loop_split requer num_hidden_layers > 0.")
    if loop_middle_layers is None:
        if num_hidden_layers % 2 != 0:
            raise ValueError(
                "enable_double_loop_split requer num_hidden_layers divisível por 2 "
                "quando loop_middle_layers não é definido."
            )
        loop_middle_layers = num_hidden_layers // 2
    if loop_middle_layers <= 0:
        raise ValueError("loop_middle_layers deve ser > 0.")
    if num_hidden_layers % loop_middle_layers != 0:
        raise ValueError("loop_middle_layers deve ser divisor de num_hidden_layers.")

    first_unlooped = (num_hidden_layers - loop_middle_layers) // 2
    middle_start = first_unlooped
    middle_end = middle_start + loop_middle_layers
    middle_repeats = (num_hidden_layers + loop_middle_layers) // loop_middle_layers
    return (
        [(idx, None) for idx in range(0, middle_start)]
        + [
            (idx, repeat_idx)
            for repeat_idx in range(middle_repeats)
            for idx in range(middle_start, middle_end)
        ]
        + [(idx, None) for idx in range(middle_end, num_hidden_layers)]
    )


class Qwen3LoopModel(Qwen3Model):
    """
    Qwen3LoopModel inherits directly from HF's Qwen3Model.
    Reuses all native Qwen3 modules (Qwen3Attention, Qwen3DecoderLayer, Qwen3RMSNorm, Qwen3MLP)
    and overrides forward() to execute the layer stack num_loops times with loop-virtual KV cache indexing.
    """
    config_class = Qwen3LoopConfig

    # ------------------------------------------------------------------
    # Helpers de loop (espelham NanbeigeModel)
    # ------------------------------------------------------------------
    def get_num_loops(self) -> int:
        if getattr(self.config, "enable_double_loop_split", False):
            return 1
        weights = getattr(self.config, "loop_loss_weights", None)
        if weights:
            return len(weights) + 1
        return getattr(self.config, "num_loops", 1)

    def get_layer_execution_order(self) -> List[Tuple[int, Optional[int]]]:
        if getattr(self.config, "enable_double_loop_split", False):
            return get_double_loop_split_layer_order(
                self.config.num_hidden_layers,
                getattr(self.config, "loop_middle_layers", None),
            )
        return [(i, None) for i in range(self.config.num_hidden_layers)]

    def get_cache_seq_length(self, past_key_values: Optional[Cache]) -> int:
        """
        O comprimento do cache tem que sair de uma camada que de fato escreveu nele.
        Com LoopSplit é a camada 0; caso contrário, o máximo entre as camadas
        virtuais que iniciam cada passagem. Espelha _get_cache_seq_length do Nanbeige.
        """
        if past_key_values is None:
            return 0
        if getattr(self.config, "enable_double_loop_split", False):
            return past_key_values.get_seq_length(0)
        max_len = 0
        for loop_idx in range(self.get_num_loops()):
            max_len = max(max_len, past_key_values.get_seq_length(loop_idx * self.config.num_hidden_layers))
        return max_len

    def _total_virtual_layers(self) -> int:
        if getattr(self.config, "enable_double_loop_split", False):
            return len(self.get_layer_execution_order())
        return self.get_num_loops() * self.config.num_hidden_layers

    def forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ) -> Union[Tuple, BaseModelOutputWithPast]:
        num_loops = self.get_num_loops()
        layer_order = self.get_layer_execution_order()
        double_split = getattr(self.config, "enable_double_loop_split", False)
        share_kv = getattr(self.config, "loop_share_kv", False)

        use_cache = use_cache if use_cache is not None else self.config.use_cache
        output_hidden_states = output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        # Gradient checkpointing e cache são incompatíveis (o cache seria reescrito
        # na recomputação do backward).
        if self.gradient_checkpointing and self.training and use_cache:
            use_cache = False

        if share_kv:
            # Compartilhar o KV da 1a passagem exige interceptar o forward da
            # atenção (o Nanbeige usa _apply_loop_shared_kv dentro do NanbeigeAttention).
            # Aqui reusamos a Qwen3Attention nativa do HF, que não tem esse gancho,
            # então a opção falha alto em vez de silenciosamente não fazer nada.
            # O relatório do Nanbeige4.2 mede ganho consistentemente menor com KV
            # compartilhado, então isto não bloqueia a paridade com os pesos publicados.
            raise NotImplementedError(
                "loop_share_kv=True exige uma subclasse de Qwen3Attention; não implementado. "
                "Os pesos publicados do Nanbeige4.2-3B usam loop_share_kv=False."
            )

        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError("You must specify exactly one of input_ids or inputs_embeds")

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        if use_cache:
            if past_key_values is None:
                past_key_values = DynamicCache(config=self.config)
            target_virtual_layers = self._total_virtual_layers()
            if hasattr(past_key_values, "layers"):
                while len(past_key_values.layers) < target_virtual_layers:
                    layer_cls = getattr(past_key_values, "layer_class_to_replicate", None)
                    if layer_cls is None:
                        from transformers.cache_utils import DynamicLayer
                        layer_cls = DynamicLayer
                    past_key_values.layers.append(layer_cls())

        past_seen_tokens = self.get_cache_seq_length(past_key_values)
        cache_position = torch.arange(past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device)
        if position_ids is None:
            position_ids = cache_position.unsqueeze(0)

        # Prepare causal mask using native HF create_causal_mask
        if not isinstance(causal_mask_mapping := attention_mask, dict):
            import inspect
            causal_params = set(inspect.signature(create_causal_mask).parameters.keys())
            embeds_key = "inputs_embeds" if "inputs_embeds" in causal_params else "input_embeds"
            mask_kwargs = {
                "config": self.config,
                embeds_key: inputs_embeds,
                "attention_mask": attention_mask,
                "past_key_values": past_key_values,
                "position_ids": position_ids,
                "cache_position": cache_position,
            }
            filtered_kwargs = {k: v for k, v in mask_kwargs.items() if k in causal_params}
            causal_mask_mapping = {
                "full_attention": create_causal_mask(**filtered_kwargs),
            }
            if getattr(self, "has_sliding_layers", False):
                sliding_params = set(inspect.signature(create_sliding_window_causal_mask).parameters.keys())
                filtered_sliding = {k: v for k, v in mask_kwargs.items() if k in sliding_params}
                causal_mask_mapping["sliding_attention"] = create_sliding_window_causal_mask(**filtered_sliding)

        hidden_states = inputs_embeds
        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        all_hidden_states = () if output_hidden_states else None
        loop_hidden_states = []
        for loop_idx in range(num_loops):
            for execution_idx, (i, _repeat_idx) in enumerate(layer_order):
                decoder_layer = self.layers[i]
                if output_hidden_states:
                    all_hidden_states = all_hidden_states + (hidden_states,)

                # Índice virtual para o KV cache. Com LoopSplit a posição na ordem de
                # execução é que define o slot; sem ele, o deslocamento por passagem.
                if double_split:
                    virtual_layer_idx = execution_idx
                else:
                    virtual_layer_idx = get_loop_cache_layer_idx(i, loop_idx, self.config.num_hidden_layers)

                orig_layer_idx = getattr(decoder_layer, "layer_idx", i)
                orig_attn_layer_idx = getattr(decoder_layer.self_attn, "layer_idx", i)
                decoder_layer.layer_idx = virtual_layer_idx
                decoder_layer.self_attn.layer_idx = virtual_layer_idx

                attn_mask = causal_mask_mapping[self.config.layer_types[i]]
                try:
                    hidden_states = decoder_layer(
                        hidden_states,
                        attention_mask=attn_mask,
                        position_embeddings=position_embeddings,
                        position_ids=position_ids,
                        past_key_values=past_key_values,
                        use_cache=use_cache,
                        **kwargs,
                    )
                finally:
                    # Restore physical layer indices
                    decoder_layer.layer_idx = orig_layer_idx
                    decoder_layer.self_attn.layer_idx = orig_attn_layer_idx

                if double_split and _repeat_idx is not None:
                    if execution_idx + 1 < len(layer_order) and layer_order[execution_idx + 1][1] != _repeat_idx:
                        loop_hidden_states.append(hidden_states)

            # Norm ao fim de cada passagem (skip_loop_final_norm=False, como no
            # Nanbeige4.2 publicado). Com True, só uma vez ao final de tudo.
            if not getattr(self.config, "skip_loop_final_norm", False):
                hidden_states = self.norm(hidden_states)

            if not double_split:
                loop_hidden_states.append(hidden_states)

            if output_hidden_states:
                all_hidden_states = all_hidden_states + (hidden_states,)

        if getattr(self.config, "skip_loop_final_norm", False):
            hidden_states = self.norm(hidden_states)
            loop_hidden_states.append(hidden_states)
            if output_hidden_states and all_hidden_states:
                all_hidden_states = all_hidden_states[:-1] + (hidden_states,)

        if not return_dict:
            return tuple(v for v in [hidden_states, past_key_values if use_cache else None, all_hidden_states] if v is not None)

        out = BaseModelOutputWithPast(
            last_hidden_state=hidden_states,
            past_key_values=past_key_values if use_cache else None,
            hidden_states=all_hidden_states,
        )
        out.loop_hidden_states = tuple(loop_hidden_states)
        return out


class Qwen3LoopForCausalLM(Qwen3ForCausalLM):
    """
    Qwen3LoopForCausalLM inherits directly from HF's Qwen3ForCausalLM.
    100% compatible with Qwen3 checkpoints and HF ecosystem.
    """
    config_class = Qwen3LoopConfig

    def __init__(self, config: Qwen3LoopConfig):
        super().__init__(config)
        self.model = Qwen3LoopModel(config)
        self.vocab_size = config.vocab_size
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.post_init()

    def _supports_default_dynamic_cache(self) -> bool:
        # Com mais de uma passagem o cache tem num_loops x num_hidden_layers
        # camadas virtuais, então o cache padrão do HF não serve.
        return self.config.num_loops == 1 and super()._supports_default_dynamic_cache()

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        logits_to_keep: int = 0,
        **kwargs,
    ) -> Union[Tuple, CausalLMOutputWithPast]:
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            **kwargs,
        )

        hidden_states = outputs[0] if not return_dict else outputs.last_hidden_state
        slice_indices = slice(-logits_to_keep, None) if isinstance(logits_to_keep, int) and logits_to_keep > 0 else slice(None)
        logits = self.lm_head(hidden_states[:, slice_indices, :])

        loss = None
        if labels is not None:
            weights = getattr(self.config, "loop_loss_weights", None)
            loop_hidden_states = getattr(outputs, "loop_hidden_states", None)
            if weights and loop_hidden_states and len(loop_hidden_states) > 1:
                padded_weights = list(weights)
                while len(padded_weights) < len(loop_hidden_states):
                    padded_weights.append(1.0)
                padded_weights = padded_weights[:len(loop_hidden_states)]

                total_loss = 0.0
                total_weight = 0.0
                for w, l_hidden in zip(padded_weights, loop_hidden_states):
                    if w > 0:
                        l_logits = self.lm_head(l_hidden[:, slice_indices, :])
                        l_loss = self.loss_function(logits=l_logits, labels=labels, vocab_size=self.config.vocab_size, **kwargs)
                        total_loss = total_loss + (w * l_loss)
                        total_weight += w
                loss = total_loss / max(1e-4, total_weight)
            else:
                loss = self.loss_function(logits=logits, labels=labels, vocab_size=self.config.vocab_size, **kwargs)

        if not return_dict:
            output = (logits,) + outputs[1:]
            return (loss,) + output if loss is not None else output

        causal_out = CausalLMOutputWithPast(
            loss=loss,
            logits=logits,
            past_key_values=outputs.past_key_values,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )
        causal_out.loop_hidden_states = getattr(outputs, "loop_hidden_states", None)
        return causal_out

    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, *model_args, **kwargs):
        loop_overrides = {
            key: kwargs.pop(key)
            for key in (
                "num_loops",
                "skip_loop_final_norm",
                "enable_double_loop_split",
                "loop_middle_layers",
                "loop_share_kv",
                "loop_loss_weights",
            )
            if key in kwargs
        }

        config = kwargs.pop("config", None)
        if config is None:
            config = AutoConfig.from_pretrained(pretrained_model_name_or_path, **kwargs)

        if not isinstance(config, Qwen3LoopConfig):
            config_dict = config.to_dict()
            config_dict.update(loop_overrides)
            config_dict.setdefault("num_loops", 2)
            config_dict["model_type"] = "qwen3loop"
            config = Qwen3LoopConfig.from_dict(config_dict)
        else:
            for key, value in loop_overrides.items():
                setattr(config, key, value)

        kwargs["config"] = config
        return super().from_pretrained(pretrained_model_name_or_path, *model_args, **kwargs)


# Register with Auto classes
try:
    AutoConfig.register("qwen3loop", Qwen3LoopConfig)
    AutoModelForCausalLM.register(Qwen3LoopConfig, Qwen3LoopForCausalLM)
except Exception:
    pass


In [ ]:
%%writefile /kaggle/working/train_dual_t4_grpo.py
"""
train_dual_t4_grpo.py
Treinamento RIC-GRPO Especializado para Kaggle Dual GPU (2x NVIDIA Tesla T4 16GB = 32GB).
Executado via: torchrun --nproc_per_node=2 train_dual_t4_grpo.py --steps 100

Otimizações Ativas para Tesla T4:
1. Precisão: torch.float16 (Turing Tensor Cores nativos) com GradScaler
2. Distribuição DDP: 2 GPUs gerando G=3 cada (G=6 total por passo) em paralelo
3. Gradient Checkpointing nativo do Qwen3Loop em cada GPU (< 3.5 GB por placa)
4. Atenção SDPA (Memory-efficient attention)
5. Sincronização de vantagens e recompensas via torch.distributed
6. Calibração de Loop 1 via RMSNorm e Sonda Latente Halting Probe
"""

import os
import sys
import json
import random
import time
import math
from typing import List, Dict, Any, Tuple
from contextlib import nullcontext

# Configurar caminhos locais
sys.path.insert(0, os.path.abspath("engine"))
sys.path.insert(0, os.path.abspath("."))

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

import qwen3loop_python  # noqa: F401
from rewards_tool_calling import (
    reward_format,
    reward_tool_selection,
    reward_arguments,
    reward_abstention,
    reward_python_execution
)

# -------------------------------------------------------------
# Configurações Globais e Hiperparâmetros
# -------------------------------------------------------------
TOTAL_STEPS = 100            # 100 passos densos com G=6 (600 rollouts)
TOTAL_GROUP_SIZE = 6         # G=6 candidatos por prompt
MAX_NEW_TOKENS = 256         # Sem truncamento de qualidade
CLIP_EPS = 0.2
GAMMA_LOOP1 = 0.30
ALPHA_DISTILL = 0.10
LAMBDA_PROBE = 0.05
LR = 2e-5
PROBE_LR = 1e-4
EVAL_INTERVAL = 25
SAVE_INTERVAL = 25

class LatentHaltingProbe(nn.Module):
    def __init__(self, hidden_size: int = 1024):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU()
        )
        self.classifier = nn.Linear(64, 3, bias=False)
        self.regressor = nn.Sequential(
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, h: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feat = self.shared(h)
        logits = self.classifier(feat)
        score = self.regressor(feat).squeeze(-1)
        return logits, score

class DynamicCurriculumSampler:
    def __init__(self, data_dir: str):
        self.l1_data = self._load_jsonl(os.path.join(data_dir, "level_1_foundation.jsonl"))
        self.l2_data = self._load_jsonl(os.path.join(data_dir, "level_2_discrimination.jsonl"))
        self.l3_data = self._load_jsonl(os.path.join(data_dir, "level_3_reasoning_python.jsonl"))
        self.eval_data = self._load_jsonl(os.path.join(data_dir, "eval_benchmark_battery.jsonl"))

    def _load_jsonl(self, filepath: str) -> List[Dict[str, Any]]:
        samples = []
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    if not line.strip(): continue
                    item = json.loads(line)
                    tools = item.get("tools")
                    if isinstance(tools, str):
                        try: tools = json.loads(tools)
                        except Exception: tools = []
                    if isinstance(tools, dict): tools = [tools]
                    if not isinstance(tools, list): tools = []
                    item["tools"] = tools
                    samples.append(item)
        return samples

    def sample_prompt(self, current_step: int, total_steps: int) -> Dict[str, Any]:
        progress = current_step / max(1, total_steps)
        if progress < 0.30:
            pool = self.l1_data if random.random() < 0.70 else self.l2_data
        elif progress < 0.70:
            r = random.random()
            if r < 0.20: pool = self.l1_data
            elif r < 0.70: pool = self.l2_data
            else: pool = self.l3_data
        else:
            r = random.random()
            if r < 0.10: pool = self.l1_data
            elif r < 0.40: pool = self.l2_data
            else: pool = self.l3_data
        return random.choice(pool)

def run_evaluation_benchmark(model, tokenizer, eval_samples: List[Dict[str, Any]], device: torch.device, max_cases: int = 15) -> Dict[str, float]:
    model.eval()
    selected = eval_samples[:max_cases]
    total = len(selected)
    correct_format, correct_selection, correct_abstention, total_abstention = 0, 0, 0, 0
    
    with torch.no_grad():
        for case in selected:
            tools = case.get("tools")
            tools_arg = tools if (isinstance(tools, list) and len(tools) > 0) else None
            prompt_str = tokenizer.apply_chat_template(case["messages"], tools=tools_arg, add_generation_prompt=True, tokenize=False)
            inputs = tokenizer(prompt_str, return_tensors="pt").to(device)
            outputs = model.generate(**inputs, max_new_tokens=160, do_sample=False, repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id)
            gen_ids = outputs[0][inputs.input_ids.shape[1]:]
            comp = tokenizer.decode(gen_ids, skip_special_tokens=False).replace("<|im_end|>", "").strip()
            
            exp_call = case.get("expected_call")
            if reward_format([comp], expected_call=[exp_call])[0] > 0.5: correct_format += 1
            if reward_tool_selection([comp], expected_call=[exp_call])[0] > 0.5: correct_selection += 1
            if exp_call is None:
                total_abstention += 1
                if reward_abstention([comp], expected_call=[None])[0] > 0.5: correct_abstention += 1

    model.train()
    return {
        "eval_format_acc": (correct_format / total) * 100,
        "eval_selection_acc": (correct_selection / total) * 100,
        "eval_abstention_acc": (correct_abstention / max(1, total_abstention)) * 100
    }

def main():
    # Inicialização DDP
    is_distributed = "RANK" in os.environ and "WORLD_SIZE" in os.environ
    if is_distributed:
        dist.init_process_group(backend="nccl")
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        device = torch.device(f"cuda:{local_rank}")
        torch.cuda.set_device(device)
    else:
        rank = 0
        local_rank = 0
        world_size = 1
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Caminhos
    MODEL_DIR = os.environ.get("MODEL_DIR", "model")
    CURRICULUM_DIR = os.environ.get("CURRICULUM_DIR", "data")
    OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "/kaggle/working/qwen3loop_tool_grpo_final")
    LOG_FILE = os.path.join(OUTPUT_DIR, "tool_grpo_training_log.json")

    # Local group size por GPU
    local_group_size = TOTAL_GROUP_SIZE // world_size

    if rank == 0:
        print("=" * 70, flush=True)
        print("INICIANDO TREINO RIC-GRPO EM DUAL GPU (KAGGLE 2x T4)", flush=True)
        print(f"GPUs Ativas:       {world_size}x Tesla T4 | Dtype: torch.float16", flush=True)
        print(f"Total Group Size:  G={TOTAL_GROUP_SIZE} ({local_group_size} por GPU)", flush=True)
        print(f"Passos Totais:     {TOTAL_STEPS} | Limite Geração: {MAX_NEW_TOKENS} tokens", flush=True)
        print("=" * 70, flush=True)
        os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Carregar Tokenizer e Modelo
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    config = AutoConfig.from_pretrained(MODEL_DIR)
    
    t_load0 = time.time()
    if rank == 0:
        print("Carregando pesos em bfloat16 nativo (mmap instantâneo)...", flush=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        config=config,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa"
    )
    if rank == 0:
        print(f"Pesos lidos em {time.time() - t_load0:.2f}s. Movendo para GPU {rank} e convertendo para float16...", flush=True)
    base_model = base_model.to(device=device, dtype=torch.float16)
    if rank == 0:
        print(f"Modelo pronto na GPU em {time.time() - t_load0:.2f}s!", flush=True)

    base_model.gradient_checkpointing_enable()

    # 2. Configurar LoRA PEFT
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    model = get_peft_model(base_model, peft_config)
    model.enable_input_require_grads()

    if is_distributed:
        model = DDP(model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=True)

    # 3. Sonda Latente
    probe = LatentHaltingProbe(config.hidden_size).to(device).to(torch.float16)
    probe_path = os.path.join(MODEL_DIR, "latent_halting_probe.pt")
    if os.path.exists(probe_path):
        try:
            probe.load_state_dict(torch.load(probe_path, map_location=device))
            if rank == 0: print("Latent Halting Probe carregada com sucesso!", flush=True)
        except Exception:
            pass

    # 4. Otimizadores e Scaler
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW([
        {"params": trainable_params, "lr": LR, "weight_decay": 0.01},
        {"params": probe.parameters(), "lr": PROBE_LR, "weight_decay": 0.01}
    ])
    scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_STEPS, eta_min=1e-6)
    scaler = GradScaler()

    # 5. Currículo
    curriculum = DynamicCurriculumSampler(CURRICULUM_DIR)
    training_history = []
    step_times = []
    start_time = time.time()

    if rank == 0:
        print("\n--- TREINAMENTO ATIVO INICIADO NO KAGGLE ---\n", flush=True)

    for step in range(1, TOTAL_STEPS + 1):
        step_t0 = time.time()
        torch.cuda.reset_peak_memory_stats(device)

        # Sincronizar o mesmo seed por passo para que ambos os ranks usem o mesmo prompt
        random.seed(42 + step)
        sample = curriculum.sample_prompt(step, TOTAL_STEPS)

        tools = sample.get("tools")
        tools_arg = tools if (isinstance(tools, list) and len(tools) > 0) else None
        prompt_str = tokenizer.apply_chat_template(sample["messages"], tools=tools_arg, add_generation_prompt=True, tokenize=False)
        prompt_inputs = tokenizer(prompt_str, return_tensors="pt").to(device)
        prompt_len = prompt_inputs.input_ids.shape[1]

        # Resetar seed local para exploração independente de cada GPU
        torch.manual_seed(1000 * (rank + 1) + step)

        # A. Rollout Paralelo: Cada GPU gera local_group_size respostas simultaneamente
        unwrapped_model = model.module if is_distributed else model
        unwrapped_model.eval()
        with torch.no_grad():
            rollout_outputs = unwrapped_model.generate(
                **prompt_inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                num_return_sequences=local_group_size,
                pad_token_id=tokenizer.eos_token_id
            )
        unwrapped_model.train()

        # Decodificar textos locais
        local_completions = []
        for i in range(local_group_size):
            gen_tokens = rollout_outputs[i][prompt_len:]
            text = tokenizer.decode(gen_tokens, skip_special_tokens=False).replace("<|im_end|>", "").strip()
            local_completions.append(text)

        # B. Coletar todas as G=6 respostas entre as GPUs para calcular recompensas e vantagens globais
        if is_distributed:
            gathered_completions = [None for _ in range(world_size)]
            dist.all_gather_object(gathered_completions, local_completions)
            all_completions = []
            for comp_list in gathered_completions:
                all_completions.extend(comp_list)
        else:
            all_completions = local_completions

        exp_call = sample.get("expected_call")
        exp_res = sample.get("expected_execution_result")

        # Avaliar recompensas hierárquicas (HTGR)
        r_fmt = reward_format(all_completions, expected_call=[exp_call]*TOTAL_GROUP_SIZE)
        r_sel = reward_tool_selection(all_completions, expected_call=[exp_call]*TOTAL_GROUP_SIZE)
        r_arg = reward_arguments(all_completions, expected_call=[exp_call]*TOTAL_GROUP_SIZE)
        r_abs = reward_abstention(all_completions, expected_call=[exp_call]*TOTAL_GROUP_SIZE)
        r_py  = reward_python_execution(all_completions, expected_execution_result=[exp_res]*TOTAL_GROUP_SIZE)

        all_rewards = [
            (rf * 1.0) + (rs * 1.0) + (ra * 1.0) + (rab * 1.5) + (rpy * 2.0)
            for rf, rs, ra, rab, rpy in zip(r_fmt, r_sel, r_arg, r_abs, r_py)
        ]

        mean_r = sum(all_rewards) / TOTAL_GROUP_SIZE
        var_r = sum((r - mean_r) ** 2 for r in all_rewards) / TOTAL_GROUP_SIZE
        std_r = math.sqrt(var_r) + 1e-4
        all_advantages = [(r - mean_r) / std_r for r in all_rewards]

        # Fatiar vantagens locais correspondentes a este rank
        local_advantages = all_advantages[rank * local_group_size : (rank + 1) * local_group_size]

        # C. Forward & Backward Pass Vetorizado
        step_ppl_l2 = 0.0
        step_ppl_l1 = 0.0
        optimizer.zero_grad()

        for i in range(local_group_size):
            seq_ids = rollout_outputs[i].unsqueeze(0)
            adv = local_advantages[i]

            sync_context = model.no_sync() if (is_distributed and i < local_group_size - 1) else nullcontext()
            with sync_context:
                with autocast(dtype=torch.float16):
                    outputs = model(input_ids=seq_ids, output_hidden_states=False, return_dict=True)

                    logits_l2 = outputs.logits[:, prompt_len - 1 : -1, :]
                    targets = seq_ids[:, prompt_len:]

                    log_probs_l2 = F.log_softmax(logits_l2, dim=-1)
                    token_log_probs_l2 = torch.gather(log_probs_l2, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
                    nll_l2 = -token_log_probs_l2.mean()
                    step_ppl_l2 += torch.exp(nll_l2.detach()).item()
                    loss_policy_l2 = nll_l2 * adv

                    # Loop 1 com RMSNorm
                    loop_hidden = getattr(outputs, "loop_hidden_states", None)
                    loss_policy_l1 = torch.tensor(0.0, device=device)
                    loss_distill = torch.tensor(0.0, device=device)

                    if loop_hidden and len(loop_hidden) > 1:
                        raw_model = unwrapped_model.base_model.model
                        h_l1_raw = loop_hidden[len(loop_hidden) // 2]
                        h_l1 = raw_model.norm(h_l1_raw)
                        logits_l1 = unwrapped_model.base_model.lm_head(h_l1[:, prompt_len - 1 : -1, :])
                        log_probs_l1 = F.log_softmax(logits_l1, dim=-1)
                        token_log_probs_l1 = torch.gather(log_probs_l1, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
                        nll_l1 = -token_log_probs_l1.mean()
                        step_ppl_l1 += torch.exp(nll_l1.detach()).item()
                        loss_policy_l1 = nll_l1 * adv

                        probs_l2_detach = F.softmax(logits_l2.detach(), dim=-1)
                        loss_distill = F.kl_div(log_probs_l1, probs_l2_detach, reduction="batchmean")

                        # Sonda
                        h_prompt_end = h_l1[:, prompt_len - 1, :]
                        probe_logits, probe_score = probe(h_prompt_end)
                        if exp_call is None: target_cls = torch.tensor([0], device=device)
                        elif "python" in exp_call.get("name", ""): target_cls = torch.tensor([2], device=device)
                        else: target_cls = torch.tensor([1], device=device)
                        loss_probe = F.cross_entropy(probe_logits, target_cls)
                    else:
                        loss_probe = torch.tensor(0.0, device=device)

                    total_sample_loss = (
                        loss_policy_l2 +
                        (GAMMA_LOOP1 * loss_policy_l1) +
                        (ALPHA_DISTILL * loss_distill) +
                        (LAMBDA_PROBE * loss_probe)
                    ) / (local_group_size * world_size)

                scaler.scale(total_sample_loss).backward()
                del outputs, total_sample_loss

        # D. Passo de Otimização com GradScaler
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params + list(probe.parameters()), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # E. Logs no Rank 0
        step_elapsed = time.time() - step_t0
        step_times.append(step_elapsed)
        peak_vram_gb = torch.cuda.max_memory_allocated(device) / (1024**3)

        avg_step_ppl_l2 = step_ppl_l2 / local_group_size
        avg_step_ppl_l1 = (step_ppl_l1 / local_group_size) if step_ppl_l1 > 0 else avg_step_ppl_l2
        phase = 1 if (step / TOTAL_STEPS) < 0.30 else (2 if (step / TOTAL_STEPS) < 0.70 else 3)

        avg_time = sum(step_times[-10:]) / len(step_times[-10:])
        eta_minutes = ((TOTAL_STEPS - step) * avg_time) / 60

        if rank == 0:
            print(
                f"[Step {step:03d}/{TOTAL_STEPS}] Fase {phase} | "
                f"Reward: {mean_r:.3f} | "
                f"PPL-L2: {avg_step_ppl_l2:.2f} | "
                f"PPL-L1: {avg_step_ppl_l1:.2f} | "
                f"VRAM T4: {peak_vram_gb:.2f} GB | "
                f"Tempo: {step_elapsed:.1f}s | "
                f"ETA: {eta_minutes:.1f} min",
                flush=True
            )
            training_history.append({
                "step": step, "phase": phase, "reward": mean_r,
                "ppl_loop2": avg_step_ppl_l2, "ppl_loop1": avg_step_ppl_l1,
                "peak_vram_gb": peak_vram_gb, "step_time_sec": step_elapsed, "eta_min": eta_minutes
            })
            with open(LOG_FILE, "w", encoding="utf-8") as f:
                json.dump(training_history, f, indent=2)

        # F. Checkpoints Periódicos
        if step % SAVE_INTERVAL == 0 and rank == 0:
            ckpt_dir = os.path.join(OUTPUT_DIR, f"checkpoint-{step}")
            os.makedirs(ckpt_dir, exist_ok=True)
            unwrapped_model.save_pretrained(ckpt_dir)
            tokenizer.save_pretrained(ckpt_dir)
            torch.save(probe.state_dict(), os.path.join(ckpt_dir, "latent_halting_probe.pt"))
            print(f"Checkpoint intermediário salvo em: {ckpt_dir}\n", flush=True)

    # 6. Finalização
    if rank == 0:
        total_time = (time.time() - start_time) / 60
        print("\n" + "=" * 70, flush=True)
        print(f"TREINAMENTO RIC-GRPO CONCLUÍDO NO KAGGLE EM {total_time:.2f} MINUTOS!", flush=True)
        unwrapped_model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        torch.save(probe.state_dict(), os.path.join(OUTPUT_DIR, "latent_halting_probe.pt"))
        print(f"Modelo final salvo em: {OUTPUT_DIR}", flush=True)
        print("=" * 70, flush=True)

    if is_distributed:
        dist.destroy_process_group()

if __name__ == "__main__":
    main()


In [ ]:
# 4. Disparar Treino Paralelo DDP em 2x Tesla T4 com streaming de saída ao vivo
!torchrun --nproc_per_node=2 /kaggle/working/train_dual_t4_grpo.py

In [ ]:
# 5. Compactar os adaptadores LoRA treinados e a sonda latente para download rápido (~40 MB)
!cd /kaggle/working && zip -r qwen3loop_tool_grpo_lora_final.zip qwen3loop_tool_grpo_final/
!ls -lh /kaggle/working/qwen3loop_tool_grpo_lora_final.zip
print('\nTreino finalizado com sucesso! Arquivo qwen3loop_tool_grpo_lora_final.zip gerado.')